## Entrenamiento en Paralelo y Métricas de Rendimiento


In [ ]:
# Paralelo
def train_subset(data):
    X_sub, y_sub = data
    model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    random_state=42
    )
    model.fit(X_sub, y_sub)
    return model

tiempos = []
accura = []
speedups = []
eficiencias = []
karp_flatts = []


NUM_PROCESOS = 7

for i in range(2, NUM_PROCESOS + 1):
  # dividir dataset
  indices = np.array_split(np.arange(len(X_train)), i)
  data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

  start_par = time.time()

  with mp.Pool(processes=i) as pool:
      models = pool.map(train_subset, data_splits)

  t_par = time.time() - start_par
  tiempos.append(t_par)

  
  p = i 
  print(f" procesadores: {p}")
  
  #1 Speed-up
  sp = t_seq / t_par 
  speedups.append(sp)
  
  #2 Eficiencia
  ef = (sp / p) * 100
  eficiencias.append(ef)
  

  print(f"Tiempo Paralelo: {t_par:.4f} seg")
  print(f"Speed-up: {sp:.2f}")
  print(f"Eficiencia: {ef:.2f}%")

acc_ind = []
for _, model in enumerate(models):
    pred_par = model.predict(X_test)
    acc_par = accuracy_score(y_test, pred_par)
    acc_ind.append(acc_par)

acc_par = np.mean(acc_ind)

tiempo_prom = np.mean(tiempos)

accura.append(acc_par)